# 03 Model Training

In [1]:
import os
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import balanced_accuracy_score, classification_report

sys.path.insert(0, os.path.abspath('..'))

from src.models.train import (
    LABEL_MAP,
    cross_validate_models,
    save_models,
    tune_lightgbm,
    tune_xgboost,
)

warnings.filterwarnings('ignore')

project_root_directory = Path('..').resolve()
processed_data_directory = project_root_directory / 'data' / 'processed'
trained_models_directory = project_root_directory / 'models'
figures_report_directory = project_root_directory / 'reports' / 'figures'

figures_report_directory.mkdir(parents=True, exist_ok=True)

training_features = pd.read_csv(processed_data_directory / 'X_train.csv')
raw_training_labels = pd.read_csv(processed_data_directory / 'y_train.csv').squeeze()

numerical_training_labels = raw_training_labels.map(LABEL_MAP).astype(int)
training_labels = raw_training_labels

print(f'Training features shape: {training_features.shape}')
print(f'Training labels shape: {training_labels.shape}')
print('\nTarget class distribution:')
print(training_labels.value_counts().to_string())

ImportError: DLL load failed while importing _devicearray: An Application Control policy has blocked this file.

14:06:44 | INFO     | Logger ready — log file: ../logs\training_20260726_140643.log


Training features shape: (577347, 53)
Training labels shape: (577347,)

Target class distribution:
class
GALAXY    377480
QSO       117143
STAR       82724


In [2]:
lightgbm_hyperparameters = tune_lightgbm(
    training_features, numerical_training_labels, n_trials=5
)
xgboost_hyperparameters = tune_xgboost(
    training_features, numerical_training_labels, n_trials=5
)

# lightgbm_hyperparameters = {
#     'learning_rate': 0.06,
#     'num_leaves': 80,
#     'max_depth': 8,
#     'min_child_samples': 25,
#     'feature_fraction': 0.8,
#     'bagging_fraction': 0.85,
#     'reg_alpha': 0.2,
#     'reg_lambda': 0.3,
# }

# xgboost_hyperparameters = {
#     'learning_rate': 0.08,
#     'max_depth': 7,
#     'subsample': 0.85,
#     'colsample_bytree': 0.8,
#     'reg_alpha': 0.1,
#     'reg_lambda': 0.5,
# }

print('Best LightGBM Parameters:')
print(lightgbm_hyperparameters)
print('\nBest XGBoost Parameters:')
print(xgboost_hyperparameters)

14:06:48 | INFO     | Tuning LightGBM (5 trials)...


  0%|          | 0/5 [00:00<?, ?it/s]

14:12:59 | INFO     | Best LightGBM score: 0.9560
14:12:59 | INFO     | Best LightGBM params: {'learning_rate': 0.07856058827501175, 'num_leaves': 147, 'max_depth': 9, 'min_child_samples': 39, 'feature_fraction': 0.6700704618321934, 'bagging_fraction': 0.932207050221457, 'bagging_freq': 9, 'reg_alpha': 0.36010545715489534, 'reg_lambda': 0.004761621554673246}
14:12:59 | INFO     | Tuning XGBoost (5 trials)...


  0%|          | 0/5 [00:00<?, ?it/s]

14:27:08 | INFO     | Best XGBoost score: 0.9555
14:27:08 | INFO     | Best XGBoost params: {'learning_rate': 0.02551869740864053, 'max_depth': 10, 'subsample': 0.6857184388823786, 'colsample_bytree': 0.7310308309205874, 'reg_alpha': 0.5703172208709745, 'reg_lambda': 0.014979938928120487, 'min_child_weight': 9}


Best LightGBM Parameters:
{'learning_rate': 0.07856058827501175, 'num_leaves': 147, 'max_depth': 9, 'min_child_samples': 39, 'feature_fraction': 0.6700704618321934, 'bagging_fraction': 0.932207050221457, 'bagging_freq': 9, 'reg_alpha': 0.36010545715489534, 'reg_lambda': 0.004761621554673246}

Best XGBoost Parameters:
{'learning_rate': 0.02551869740864053, 'max_depth': 10, 'subsample': 0.6857184388823786, 'colsample_bytree': 0.7310308309205874, 'reg_alpha': 0.5703172208709745, 'reg_lambda': 0.014979938928120487, 'min_child_weight': 9}


In [3]:
(
    lightgbm_models,
    xgboost_models,
    catboost_models,
    mlp_models,
    out_of_fold_predictions_lightgbm,
    out_of_fold_predictions_xgboost,
    out_of_fold_predictions_catboost,
    out_of_fold_predictions_mlp,
) = cross_validate_models(
    training_features,
    training_labels,
    n_splits=10,
    lgb_params=lightgbm_hyperparameters,
    xgb_params=xgboost_hyperparameters,
    include_mlp=False,
)

print('Cross-validation completed successfully.')

14:27:08 | INFO     | Starting 10-Fold CV | Train size: 577,347
14:27:08 | INFO     | =======================================================
14:27:08 | INFO     | FOLD 1/10
14:27:08 | INFO     | =======================================================
14:27:08 | INFO     | Train: 519,612 | Val: 57,735
14:27:08 | INFO     | Training LightGBM...


[100]	valid_0's multi_logloss: 0.0992855
[200]	valid_0's multi_logloss: 0.0972058


14:27:32 | INFO     | LightGBM BA: 0.9659
14:27:32 | INFO     | Training XGBoost...


[0]	validation_0-mlogloss:0.84716
[100]	validation_0-mlogloss:0.14157
[200]	validation_0-mlogloss:0.09949
[300]	validation_0-mlogloss:0.09245
[400]	validation_0-mlogloss:0.09013
[500]	validation_0-mlogloss:0.08922
[600]	validation_0-mlogloss:0.08878
[700]	validation_0-mlogloss:0.08848
[800]	validation_0-mlogloss:0.08841
[900]	validation_0-mlogloss:0.08832
[993]	validation_0-mlogloss:0.08827


14:29:35 | INFO     | XGBoost BA: 0.9550
14:29:35 | INFO     | Training CatBoost...


...

0:	learn: 0.9196715	test: 0.9182549	best: 0.9182549 (0)	total: 349ms	remaining: 11m 38s
100:	learn: 0.9529565	test: 0.9528946	best: 0.9528946 (100)	total: 35.8s	remaining: 11m 13s
200:	learn: 0.9587340	test: 0.9583402	best: 0.9584355 (197)	total: 1m 11s	remaining: 10m 37s
300:	learn: 0.9609847	test: 0.9604721	best: 0.9605166 (299)	total: 1m 46s	remaining: 10m 2s
400:	learn: 0.9626093	test: 0.9617768	best: 0.9617769 (397)	total: 2m 21s	remaining: 9m 26s
500:	learn: 0.9637244	test: 0.9623107	best: 0.9623107 (500)	total: 2m 57s	remaining: 8m 50s
600:	learn: 0.9645704	test: 0.9625747	best: 0.9627595 (578)	total: 3m 32s	remaining: 8m 14s


15:34:16 | INFO     | CatBoost BA: 0.9628
15:34:16 | INFO     | Fold 10 Ensemble BA: 0.9645


Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.9627594747
bestIteration = 578

Shrink model to first 579 iterations.


15:34:17 | INFO     | =======================================================
15:34:17 | INFO     | FINAL CV RESULTS
15:34:17 | INFO     | LightGBM  OOF: 0.9647
15:34:17 | INFO     | XGBoost   OOF: 0.9568
15:34:17 | INFO     | CatBoost  OOF: 0.9624
15:34:17 | INFO     | Ensemble  OOF: 0.9638
15:34:17 | INFO     | =======================================================


Cross-validation completed successfully.


In [4]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold

LABEL_MAP = {'GALAXY': 0, 'QSO': 1, 'STAR': 2}
encoded_training_labels = training_labels.map(LABEL_MAP).astype(int)
stratified_kfold_cross_validator = StratifiedKFold(
    n_splits=10, shuffle=True, random_state=42
)

model_fold_balanced_accuracy_scores = {
    'LightGBM': [],
    'XGBoost': [],
    'CatBoost': [],
    'Ensemble': [],
}

for _, validation_indices in stratified_kfold_cross_validator.split(
    training_features, encoded_training_labels
):
    true_validation_labels = encoded_training_labels.iloc[validation_indices]

    predicted_validation_labels_lightgbm = out_of_fold_predictions_lightgbm[
        validation_indices
    ].argmax(axis=1)
    predicted_validation_labels_xgboost = out_of_fold_predictions_xgboost[
        validation_indices
    ].argmax(axis=1)
    predicted_validation_labels_catboost = out_of_fold_predictions_catboost[
        validation_indices
    ].argmax(axis=1)

    model_fold_balanced_accuracy_scores['LightGBM'].append(
        balanced_accuracy_score(
            true_validation_labels, predicted_validation_labels_lightgbm
        )
    )
    model_fold_balanced_accuracy_scores['XGBoost'].append(
        balanced_accuracy_score(
            true_validation_labels, predicted_validation_labels_xgboost
        )
    )
    model_fold_balanced_accuracy_scores['CatBoost'].append(
        balanced_accuracy_score(
            true_validation_labels, predicted_validation_labels_catboost
        )
    )

    validation_ensemble_probabilities = (
        out_of_fold_predictions_lightgbm[validation_indices] * 0.60
        + out_of_fold_predictions_xgboost[validation_indices] * 0.35
        + out_of_fold_predictions_catboost[validation_indices] * 0.05
    )
    predicted_validation_labels_ensemble = (
        validation_ensemble_probabilities.argmax(axis=1)
    )

    model_fold_balanced_accuracy_scores['Ensemble'].append(
        balanced_accuracy_score(
            true_validation_labels, predicted_validation_labels_ensemble
        )
    )

fold_scores_dataframe = pd.DataFrame(
    model_fold_balanced_accuracy_scores, index=range(1, 11)
)

plt.figure(figsize=(10, 5))
sns.set_style('whitegrid')
sns.lineplot(data=fold_scores_dataframe, markers=True, dashes=False)
plt.title('Balanced Accuracy per Fold (10-Fold Stratified CV)')
plt.xlabel('Fold')
plt.ylabel('Balanced Accuracy')
plt.xticks(range(1, 11))
plt.ylim(0.94, 0.965)
plt.tight_layout()
plt.savefig(
    figures_report_directory / 'oof_balanced_accuracy_per_fold.png', dpi=200
)
plt.close()

ensemble_out_of_fold_probabilities = (
    out_of_fold_predictions_lightgbm * 0.60
    + out_of_fold_predictions_xgboost * 0.35
    + out_of_fold_predictions_catboost * 0.05
)

overall_out_of_fold_balanced_accuracy_scores = {
    'LightGBM': balanced_accuracy_score(
        encoded_training_labels,
        out_of_fold_predictions_lightgbm.argmax(axis=1),
    ),
    'XGBoost': balanced_accuracy_score(
        encoded_training_labels, out_of_fold_predictions_xgboost.argmax(axis=1)
    ),
    'CatBoost': balanced_accuracy_score(
        encoded_training_labels,
        out_of_fold_predictions_catboost.argmax(axis=1),
    ),
    'Ensemble': balanced_accuracy_score(
        encoded_training_labels,
        ensemble_out_of_fold_probabilities.argmax(axis=1),
    ),
}

plt.figure(figsize=(8, 5))
model_performance_barplot_axis = sns.barplot(
    x=list(overall_out_of_fold_balanced_accuracy_scores.keys()),
    y=list(overall_out_of_fold_balanced_accuracy_scores.values()),
    palette=['#4C72B0', '#DD8452', '#55A868', '#C44E52'],
)

for bar_index, balanced_accuracy_value in enumerate(
    overall_out_of_fold_balanced_accuracy_scores.values()
):
    model_performance_barplot_axis.text(
        bar_index,
        balanced_accuracy_value + 0.0004,
        f'{balanced_accuracy_value:.4f}',
        ha='center',
        va='bottom',
        fontsize=10,
    )

plt.title('Final OOF Balanced Accuracy: Base Models vs. Ensemble')
plt.ylabel('Balanced Accuracy')
plt.ylim(0.95, 0.965)
plt.tight_layout()
plt.savefig(
    figures_report_directory / 'final_oof_balanced_accuracy.png', dpi=200
)
plt.close()

In [5]:
save_models(
    lightgbm_models,
    xgboost_models,
    catboost_models,
    output_dir=str(trained_models_directory),
)

print(f'Saved base models to {trained_models_directory}')

15:34:28 | INFO     | Saved 10 LGB + 10 XGB + 10 CAT models to 'D:\Dev\predicting-stellar-class\models'


Saved base models to D:\Dev\predicting-stellar-class\models


In [6]:
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support

CLASS_LABEL_MAP = {'GALAXY': 0, 'QSO': 1, 'STAR': 2}
class_names = list(CLASS_LABEL_MAP.keys())

encoded_training_labels = training_labels.map(CLASS_LABEL_MAP).astype(int)

model_predicted_probabilities = {
    'LightGBM': out_of_fold_predictions_lightgbm,
    'XGBoost': out_of_fold_predictions_xgboost,
    'CatBoost': out_of_fold_predictions_catboost,
    'Ensemble': ensemble_out_of_fold_probabilities,
}

print('Per-model classification reports:')
for model_name, class_probabilities in model_predicted_probabilities.items():
    predicted_class_labels = class_probabilities.argmax(axis=1)

    print(f'\n--- {model_name} ---')
    print(
        classification_report(
            encoded_training_labels,
            predicted_class_labels,
            target_names=class_names,
        )
    )

    evaluation_confusion_matrix = confusion_matrix(
        encoded_training_labels, predicted_class_labels
    )
    print('Confusion matrix:')
    print(evaluation_confusion_matrix)

for model_name, class_probabilities in model_predicted_probabilities.items():
    predicted_class_labels = class_probabilities.argmax(axis=1)
    evaluation_confusion_matrix = confusion_matrix(
        encoded_training_labels, predicted_class_labels
    )

    plt.figure(figsize=(5, 4))
    sns.heatmap(
        evaluation_confusion_matrix,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=class_names,
        yticklabels=class_names,
    )
    plt.title(f'{model_name} Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.tight_layout()
    plt.savefig(
        figures_report_directory
        / f'{model_name.lower()}_confusion_matrix.png',
        dpi=200,
    )
    plt.close()

for fold_number, model_instance in enumerate(lightgbm_models, start=1):
    feature_importance_gains = model_instance.feature_importance(
        importance_type='gain'
    )
    top_feature_indices = np.argsort(feature_importance_gains)[-15:][::-1]

    top_feature_names = training_features.columns[
        top_feature_indices
    ].tolist()
    top_feature_gains = feature_importance_gains[top_feature_indices]

    plt.figure(figsize=(8, 4))
    plt.barh(top_feature_names, top_feature_gains)
    plt.title(f'LightGBM Feature Importance (Fold {fold_number})')
    plt.tight_layout()
    plt.savefig(
        figures_report_directory
        / f'lgb_feature_importance_fold{fold_number}.png',
        dpi=200,
    )
    plt.close()


per_class_recall_by_model = {}
for model_name, class_probabilities in model_predicted_probabilities.items():
    predicted_class_labels = class_probabilities.argmax(axis=1)
    _, per_class_recall, _, _ = precision_recall_fscore_support(
        encoded_training_labels,
        predicted_class_labels,
        labels=[0, 1, 2],
    )
    per_class_recall_by_model[model_name] = per_class_recall

per_class_recall_dataframe = pd.DataFrame(
    per_class_recall_by_model, index=class_names
)

recall_chart_axis = per_class_recall_dataframe.plot(
    kind='bar',
    figsize=(9, 6),
    color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'],
    width=0.8,
)
plt.title('Per-Class Recall by Model (OOF)')
plt.xlabel('Class')
plt.ylabel('Recall')
plt.xticks(rotation=0)
plt.ylim(0.75, 1.00)
plt.legend(title='Model')
plt.tight_layout()
plt.savefig(
    figures_report_directory / 'per_class_recall_by_model.png', dpi=200
)
plt.close()

print(f'Saved evaluation visualizations to {figures_report_directory}')

Per-model classification reports:

--- LightGBM ---
              precision    recall  f1-score   support

      GALAXY       0.99      0.96      0.97    377480
         QSO       0.95      0.97      0.96    117143
        STAR       0.87      0.96      0.91     82724

    accuracy                           0.96    577347
   macro avg       0.94      0.96      0.95    577347
weighted avg       0.96      0.96      0.96    577347

Confusion matrix:
[[361402   5166  10912]
 [  1923 114065   1155]
 [  2662    411  79651]]

--- XGBoost ---
              precision    recall  f1-score   support

      GALAXY       0.98      0.98      0.98    377480
         QSO       0.97      0.97      0.97    117143
        STAR       0.93      0.93      0.93     82724

    accuracy                           0.97    577347
   macro avg       0.96      0.96      0.96    577347
weighted avg       0.97      0.97      0.97    577347

Confusion matrix:
[[369163   3287   5030]
 [  3109 113054    980]
 [  5543    

## OOF-Weighted Blend & Submission

In [7]:
test_features = pd.read_csv(processed_data_directory / 'X_test.csv')
raw_test_data = pd.read_csv(
    project_root_directory / 'data' / 'raw' / 'test.csv'
)

test_predictions_lightgbm = np.zeros((len(test_features), 3))
for lightgbm_model in lightgbm_models:
    test_predictions_lightgbm += lightgbm_model.predict_proba(test_features)
test_predictions_lightgbm /= len(lightgbm_models)

test_predictions_xgboost = np.zeros((len(test_features), 3))
for xgboost_model in xgboost_models:
    test_predictions_xgboost += xgboost_model.predict_proba(test_features)
test_predictions_xgboost /= len(xgboost_models)

test_predictions_catboost = np.zeros((len(test_features), 3))
for catboost_model in catboost_models:
    test_predictions_catboost += catboost_model.predict_proba(test_features)
test_predictions_catboost /= len(catboost_models)

print(
    f'Test predictions generated: LGB {test_predictions_lightgbm.shape}, '
    f'XGB {test_predictions_xgboost.shape}, CAT {test_predictions_catboost.shape}'
)

Test predictions generated: LGB (247435, 3), XGB (247435, 3), CAT (247435, 3)


In [8]:
highest_balanced_accuracy = -np.inf
optimal_model_weights = (1 / 3, 1 / 3, 1 / 3)

for weight_lightgbm in np.arange(0.0, 1.01, 0.05):
    for weight_xgboost in np.arange(0.0, 1.01 - weight_lightgbm, 0.05):
        weight_catboost = 1.0 - weight_lightgbm - weight_xgboost
        if weight_catboost < -1e-9:
            continue
        weight_catboost = max(weight_catboost, 0.0)

        blended_out_of_fold_probabilities = (
            out_of_fold_predictions_lightgbm * weight_lightgbm
            + out_of_fold_predictions_xgboost * weight_xgboost
            + out_of_fold_predictions_catboost * weight_catboost
        )
        current_score = balanced_accuracy_score(
            encoded_training_labels,
            blended_out_of_fold_probabilities.argmax(axis=1),
        )

        if current_score > highest_balanced_accuracy:
            highest_balanced_accuracy = current_score
            optimal_model_weights = (
                round(weight_lightgbm, 2),
                round(weight_xgboost, 2),
                round(weight_catboost, 2),
            )

print(f"Best weights (LGB, XGB, CAT) = {optimal_model_weights}")
print(f"Best OOF Balanced Accuracy   = {highest_balanced_accuracy:.5f}")

(
    optimal_weight_lightgbm,
    optimal_weight_xgboost,
    optimal_weight_catboost,
) = optimal_model_weights

blended_out_of_fold_predictions = (
    out_of_fold_predictions_lightgbm * optimal_weight_lightgbm
    + out_of_fold_predictions_xgboost * optimal_weight_xgboost
    + out_of_fold_predictions_catboost * optimal_weight_catboost
)

blended_test_predictions = (
    test_predictions_lightgbm * optimal_weight_lightgbm
    + test_predictions_xgboost * optimal_weight_xgboost
    + test_predictions_catboost * optimal_weight_catboost
)

Best weights (LGB, XGB, CAT) = (np.float64(0.85), np.float64(0.0), np.float64(0.15))
Best OOF Balanced Accuracy   = 0.96476


In [9]:
INDEX_TO_CLASS_LABEL_MAP = {0: 'GALAXY', 1: 'QSO', 2: 'STAR'}

predicted_test_class_indices = blended_test_predictions.argmax(axis=1)
predicted_test_class_labels = np.array(
    [INDEX_TO_CLASS_LABEL_MAP[index] for index in predicted_test_class_indices]
)

submission_dataframe = pd.DataFrame(
    {'id': raw_test_data['id'], 'class': predicted_test_class_labels}
)

submission_output_path = (
    project_root_directory / 'data' / 'processed' / 'submission7.csv'
)
submission_dataframe.to_csv(submission_output_path, index=False)

class_label_distribution = submission_dataframe['class'].value_counts().to_dict()

print(f"Submission saved: {submission_output_path}")
print(f"Class distribution: {class_label_distribution}")

submission_dataframe.head()

Submission saved: D:\Dev\predicting-stellar-class\data\processed\submission7.csv
Class distribution: {'GALAXY': 156855, 'QSO': 51247, 'STAR': 39333}


,id,class
0,577347,GALAXY
1,577348,GALAXY
2,577349,GALAXY
3,577350,STAR
4,577351,GALAXY
